In [1]:
# 03_wheat_dag.ipynb

In [2]:
# Imports and paths

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("../..")   # change if needed
PROCESSED = BASE / "data" / "processed" / "CY-Bench"
RESULTS = BASE / "data" / "results" / "CY-Bench"
RESULTS.mkdir(parents=True, exist_ok=True)

panel_path = PROCESSED / "wheat_IN_panel_full_year.csv"
panel = pd.read_csv(panel_path)

panel.shape

(7552, 45)

In [3]:
# columns relevant for DAG work

candidate_cols = [
    "crop_name", "country_code", "adm_id", "harvest_year",
    "yield", "production", "harvest_area",
    "avg_ssm", "avg_rsm", "avg_ndvi", "avg_fpar",
    "avg_tmin", "avg_tmax", "avg_tavg", "avg_prec", "sum_prec",
    "avg_rad", "avg_et0", "avg_vpd", "avg_cwb",
    "awc", "bulk_density", "drainage_class",
    "latitude", "longitude", "region_area",
    "crop_area", "crop_area_percentage",
    "sos", "eos"
]

dag_df = panel[[c for c in candidate_cols if c in panel.columns]].copy()
dag_df.shape

(7552, 30)

In [4]:
# Basic missingness and descriptive summary

missing = (
    dag_df.isna().mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing.head(30)

,column,missing_fraction
0,avg_rsm,0.110832
1,avg_ssm,0.110832
2,adm_id,0.000000
3,harvest_year,0.000000
4,crop_name,0.000000
5,country_code,0.000000
6,production,0.000000
7,yield,0.000000
8,harvest_area,0.000000
9,avg_ndvi,0.000000


In [5]:
# Choosing a first DAG variable set

dag_vars = [
    "yield",
    "avg_ssm",
    "avg_rsm",
    "avg_ndvi",
    "avg_fpar",
    "avg_tavg",
    "avg_prec",
    "avg_vpd",
    "avg_cwb",
    "awc",
    "bulk_density",
    "drainage_class",
    "latitude",
    "longitude",
    "region_area",
    "harvest_area",
]

dag_vars = [c for c in dag_vars if c in dag_df.columns]
dag_vars

['yield',
 'avg_ssm',
 'avg_rsm',
 'avg_ndvi',
 'avg_fpar',
 'avg_tavg',
 'avg_prec',
 'avg_vpd',
 'avg_cwb',
 'awc',
 'bulk_density',
 'drainage_class',
 'latitude',
 'longitude',
 'region_area',
 'harvest_area']

In [6]:
# Correlation screen for quick orientation only
# This is not causal discovery, just a rough diagnostic.

num_df = dag_df[dag_vars].select_dtypes(include=[np.number]).copy()
corr = num_df.corr(numeric_only=True)

corr

,yield,avg_ssm,avg_rsm,avg_ndvi,avg_fpar,avg_tavg,avg_prec,avg_vpd,avg_cwb,awc,bulk_density,drainage_class,latitude,longitude,region_area,harvest_area
yield,1.000000,-0.513086,-0.496648,-0.089134,0.181229,-0.024063,-0.380310,0.191194,-0.332135,0.038464,0.027305,0.351697,0.536883,-0.364752,-0.198527,0.612232
avg_ssm,-0.513086,1.000000,0.913911,0.488205,0.182696,-0.229715,0.759863,-0.601272,0.753508,0.093045,-0.279049,-0.414570,-0.240521,0.553336,-0.143965,-0.429743
avg_rsm,-0.496648,0.913911,1.000000,0.274111,0.017374,-0.079658,0.571383,-0.325587,0.528910,0.151638,-0.100162,-0.466493,-0.402568,0.287032,0.024655,-0.403391
avg_ndvi,-0.089134,0.488205,0.274111,1.000000,0.798570,-0.480297,0.598991,-0.736396,0.687821,0.088462,-0.359599,-0.161896,0.192074,0.505474,-0.426688,-0.112717
avg_fpar,0.181229,0.182696,0.017374,0.798570,1.000000,-0.378071,0.280771,-0.467915,0.378656,-0.006841,-0.163912,0.102899,0.342943,0.280273,-0.395087,0.106414
avg_tavg,-0.024063,-0.229715,-0.079658,-0.480297,-0.378071,1.000000,-0.255905,0.625382,-0.379332,-0.000806,0.393506,-0.162301,-0.544868,-0.094900,0.214963,-0.041758
avg_prec,-0.380310,0.759863,0.571383,0.598991,0.280771,-0.255905,1.000000,-0.727063,0.983029,0.029216,-0.383528,-0.286136,-0.004843,0.732051,-0.237650,-0.354270
avg_vpd,0.191194,-0.601272,-0.325587,-0.736396,-0.467915,0.625382,-0.727063,1.000000,-0.820556,-0.105492,0.530621,0.098142,-0.284955,-0.685534,0.388385,0.202429
avg_cwb,-0.332135,0.753508,0.528910,0.687821,0.378656,-0.379332,0.983029,-0.820556,1.000000,0.029294,-0.431530,-0.243801,0.102222,0.758820,-0.319183,-0.314099
awc,0.038464,0.093045,0.151638,0.088462,-0.006841,-0.000806,0.029216,-0.105492,0.029294,1.000000,-0.135496,-0.160630,0.019372,0.008263,-0.102701,0.156481


In [7]:
# First wheat DAG as a DOT string
# Domain-driven structure, not learned from correlation alone.

wheat_dag_dot = r"""
digraph {
    awc -> avg_ssm;
    awc -> avg_rsm;
    bulk_density -> avg_ssm;
    bulk_density -> avg_rsm;
    drainage_class -> avg_ssm;
    drainage_class -> avg_rsm;

    avg_prec -> avg_ssm;
    avg_prec -> avg_rsm;
    avg_prec -> avg_cwb;
    avg_tavg -> avg_et0;
    avg_et0 -> avg_cwb;
    avg_vpd -> avg_fpar;
    avg_vpd -> avg_ndvi;

    avg_ssm -> avg_ndvi;
    avg_ssm -> avg_fpar;
    avg_rsm -> avg_ndvi;
    avg_rsm -> avg_fpar;

    avg_tavg -> avg_ndvi;
    avg_tavg -> avg_fpar;
    avg_cwb -> avg_ndvi;
    avg_cwb -> avg_fpar;

    avg_ndvi -> yield;
    avg_fpar -> yield;
    avg_ssm -> yield;
    avg_rsm -> yield;
    avg_prec -> yield;
    avg_tavg -> yield;
    avg_cwb -> yield;

    latitude -> avg_tavg;
    latitude -> avg_prec;
    longitude -> avg_tavg;
    longitude -> avg_prec;
    region_area -> harvest_area;

    harvest_area -> production;
    yield -> production;
}
"""

print(wheat_dag_dot)


digraph {
    awc -> avg_ssm;
    awc -> avg_rsm;
    bulk_density -> avg_ssm;
    bulk_density -> avg_rsm;
    drainage_class -> avg_ssm;
    drainage_class -> avg_rsm;

    avg_prec -> avg_ssm;
    avg_prec -> avg_rsm;
    avg_prec -> avg_cwb;
    avg_tavg -> avg_et0;
    avg_et0 -> avg_cwb;
    avg_vpd -> avg_fpar;
    avg_vpd -> avg_ndvi;

    avg_ssm -> avg_ndvi;
    avg_ssm -> avg_fpar;
    avg_rsm -> avg_ndvi;
    avg_rsm -> avg_fpar;

    avg_tavg -> avg_ndvi;
    avg_tavg -> avg_fpar;
    avg_cwb -> avg_ndvi;
    avg_cwb -> avg_fpar;

    avg_ndvi -> yield;
    avg_fpar -> yield;
    avg_ssm -> yield;
    avg_rsm -> yield;
    avg_prec -> yield;
    avg_tavg -> yield;
    avg_cwb -> yield;

    latitude -> avg_tavg;
    latitude -> avg_prec;
    longitude -> avg_tavg;
    longitude -> avg_prec;
    region_area -> harvest_area;

    harvest_area -> production;
    yield -> production;
}



In [8]:
# Save DAG text for later DoWhy use

dag_path = RESULTS / "wheat_dag.dot"
with open(dag_path, "w") as f:
    f.write(wheat_dag_dot)

dag_path

PosixPath('../../data/results/CY-Bench/wheat_dag.dot')

In [9]:
# Optional: simple edge table

edges = []
for line in wheat_dag_dot.splitlines():
    line = line.strip()
    if "->" in line and line.endswith(";"):
        src, dst = line.replace(";", "").split("->")
        edges.append({"source": src.strip(), "target": dst.strip()})

edge_df = pd.DataFrame(edges)
edge_df

,source,target
0,awc,avg_ssm
1,awc,avg_rsm
2,bulk_density,avg_ssm
3,bulk_density,avg_rsm
4,drainage_class,avg_ssm
5,drainage_class,avg_rsm
6,avg_prec,avg_ssm
7,avg_prec,avg_rsm
8,avg_prec,avg_cwb
9,avg_tavg,avg_et0


In [10]:
# Save edge list

edge_path = RESULTS / "wheat_dag_edges.csv"
edge_df.to_csv(edge_path, index=False)

edge_path

PosixPath('../../data/results/CY-Bench/wheat_dag_edges.csv')

In [11]:
# Placeholder for CCM-informed DAG revision after CCM completes

ccm_best_path = RESULTS / "wheat_IN_ccm_best.csv"

if ccm_best_path.exists():
    ccm_best = pd.read_csv(ccm_best_path)
    ccm_best
else:
    print("CCM results not available yet. Keep the DAG domain-driven for now.")

In [12]:
# Notes for interpretation

dag_notes = {
    "outcome": "yield",
    "water_state_nodes": ["avg_ssm", "avg_rsm", "avg_cwb"],
    "vegetation_nodes": ["avg_ndvi", "avg_fpar"],
    "climate_nodes": ["avg_prec", "avg_tavg", "avg_vpd", "avg_et0"],
    "soil_nodes": ["awc", "bulk_density", "drainage_class"],
    "spatial_nodes": ["latitude", "longitude", "region_area"],
}

dag_notes

{'outcome': 'yield',
 'water_state_nodes': ['avg_ssm', 'avg_rsm', 'avg_cwb'],
 'vegetation_nodes': ['avg_ndvi', 'avg_fpar'],
 'climate_nodes': ['avg_prec', 'avg_tavg', 'avg_vpd', 'avg_et0'],
 'soil_nodes': ['awc', 'bulk_density', 'drainage_class'],
 'spatial_nodes': ['latitude', 'longitude', 'region_area']}